Day 4 — LLM Explanation Layer & Agent
**Predictive Denial Prevention AI Platform**

This notebook implements:

- Tool 1: `get_patient_from_fhir(patient_id)` — fetches patient + claim from HAPI FHIR, maps to ML feature vector
- Tool 2: `predict_fraud_risk(feature_vector)` — runs trained ML model, returns risk score + SHAP values
- LLM Orchestrator — calls `databricks-meta-llama-3-3-70b-instruct` with structured JSON output
- MLflow LLM Experiment — logs prompt version, token counts, latency, SHAP artifacts
- ChatAgent — registers the full pipeline as a deployable agent for AI Playground
- Delta sink — writes every prediction to a Delta table for the analytics dashboard


## 0 · Install Dependencies


In [ ]:
# %pip install databricks-openai shap requests mlflow --upgrade -q
# dbutils.library.restartPython()

1 · Configuration
**All tuneable constants live here. Nothing below this cell contains hard-coded values.**


In [ ]:
# ── Workspace / endpoints ─────────────────────────────────────────────────────
DATABRICKS_HOST = dbutils.notebook.entry_point.getDbutils(
).notebook().getContext().apiUrl().get()

FHIR_BASE_URL = "https://hapi.fhir.org/baseR4"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

UC_CATALOG = "<your_catalog>"
UC_SCHEMA = "<your_schema>"
ML_MODEL_REGISTRY_NAME = "<your_registered_model_name>"

PROVIDER_LOOKUP_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.provider_lookup"
PREDICTIONS_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.llm_predictions"

LLM_EXPERIMENT_NAME = "/Shared/denial_prevention_llm"
PROMPT_TEMPLATE_VERSION = "v1.1"
TOP_N_SHAP = 3
MOCK_MODE = False

2 · MLflow Experiment Setup


In [ ]:
import mlflow

#  Separate experiment from the ML training runs — do NOT share
mlflow.set_experiment(LLM_EXPERIMENT_NAME)

#  Enable automatic tracing of OpenAI-compatible calls:
#  captures token counts, latency, and full prompt/response as MLflow traces
mlflow.openai.autolog()

print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Experiment          : {LLM_EXPERIMENT_NAME}")

3 · Tool 1 — `get_patient_from_fhir`

Fetches Patient + Claim resources from HAPI FHIR, then maps FHIR fields
to the exact feature names expected by the trained ML model.

**FHIR → Kaggle feature mapping** (per architecture doc):
| FHIR Field | ML Feature |
|---|---|
| `Patient.birthDate` | `Age` |
| `Condition[].code.coding` | `ChronicCond_*` flags |
| `Claim.total.value` | `InscClaimAmtReimbursed` |
| `Claim.diagnosis[].diagnosisCodeableConcept` | `ClmDiagnosisCode_1..9` |
| `Claim.procedure[].procedureCodeableConcept` | `ClmProcedureCode_1..2` |
| `Patient.gender` | `Gender` (1=Male, 0=Female) |


In [ ]:
import json
import time
import requests
import pandas as pd
import numpy as np
import mlflow
import shap

from datetime import date, datetime
from typing import Optional
from pyspark.sql import functions as F


# ── ICD-10 chapter prefix → ChronicCond_* flag mapping ───────────────────────
ICD_TO_CHRONIC_FLAG = {
    # Alzheimer / Dementia
    "G30": "ChronicCond_Alzheimer",
    "F00": "ChronicCond_Alzheimer",
    "F01": "ChronicCond_Alzheimer",
    "F02": "ChronicCond_Alzheimer",
    "F03": "ChronicCond_Alzheimer",

    # Heart Failure
    "I50": "ChronicCond_Heartfailure",

    # Kidney Disease / Renal
    "N18": "ChronicCond_KidneyDisease",
    "N19": "ChronicCond_KidneyDisease",

    # Cancer
    "C18": "ChronicCond_Cancer",
    "C34": "ChronicCond_Cancer",
    "C50": "ChronicCond_Cancer",

    # COPD
    "J44": "ChronicCond_ObstrPulmonary",

    # Depression
    "F32": "ChronicCond_Depression",
    "F33": "ChronicCond_Depression",

    # Diabetes
    "E10": "ChronicCond_Diabetes",
    "E11": "ChronicCond_Diabetes",
    "E13": "ChronicCond_Diabetes",

    # Ischemic Heart Disease
    "I25": "ChronicCond_IschemicHeart",

    # Osteoporosis
    "M80": "ChronicCond_Osteoporasis",
    "M81": "ChronicCond_Osteoporasis",

    # Rheumatoid arthritis / osteoarthritis proxy bucket
    "M05": "ChronicCond_rheumatoidarthritis",
    "M06": "ChronicCond_rheumatoidarthritis",
    "M15": "ChronicCond_rheumatoidarthritis",

    # Stroke / TIA
    "I63": "ChronicCond_stroke",
    "I64": "ChronicCond_stroke",
    "G45": "ChronicCond_stroke",
}

ALL_CHRONIC_FLAGS = [
    "ChronicCond_Alzheimer",
    "ChronicCond_Heartfailure",
    "ChronicCond_KidneyDisease",
    "ChronicCond_Cancer",
    "ChronicCond_ObstrPulmonary",
    "ChronicCond_Depression",
    "ChronicCond_Diabetes",
    "ChronicCond_IschemicHeart",
    "ChronicCond_Osteoporasis",
    "ChronicCond_rheumatoidarthritis",
    "ChronicCond_stroke",
]


def _fhir_get(
    resource_type: str,
    params: dict | None = None,
    resource_id: str | None = None,
    max_retries: int = 3,
    base_sleep_seconds: float = 1.5,
) -> dict:
    """
    Generic FHIR GET helper with simple retry/backoff for public sandbox stability.
    """
    if resource_id:
        url = f"{FHIR_BASE_URL}/{resource_type}/{resource_id}"
    else:
        url = f"{FHIR_BASE_URL}/{resource_type}"

    headers = {"Accept": "application/fhir+json"}

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, params=params,
                                timeout=20, headers=headers)

            # Retry only for transient errors
            if resp.status_code in {429, 500, 502, 503, 504}:
                raise requests.HTTPError(
                    f"Transient FHIR error {resp.status_code}: {resp.text[:300]}",
                    response=resp,
                )

            resp.raise_for_status()
            return resp.json()

        except Exception as e:
            last_err = e
            if attempt == max_retries:
                break
            sleep_s = base_sleep_seconds * (2 ** (attempt - 1))
            print(
                f"[WARN] FHIR GET failed (attempt {attempt}/{max_retries}): {e}")
            time.sleep(sleep_s)

    raise RuntimeError(
        f"FHIR GET failed after {max_retries} attempts: {last_err}")


def _compute_age(birth_date_str: str, reference_date: Optional[date] = None) -> int:
    """
    Compute integer age from a FHIR birthDate string (YYYY-MM-DD),
    using claim date as reference when available.
    """
    ref = reference_date or date.today()
    bd = datetime.strptime(birth_date_str, "%Y-%m-%d").date()

    years = ref.year - bd.year
    if (ref.month, ref.day) < (bd.month, bd.day):
        years -= 1
    return max(years, 0)


def _map_chronic_conditions(conditions: list[dict]) -> dict:
    """
    Given FHIR Condition resources, return ChronicCond_* binary flags.
    """
    flags = {flag: 0 for flag in ALL_CHRONIC_FLAGS}

    for cond in conditions:
        for coding in cond.get("code", {}).get("coding", []):
            code_prefix = (coding.get("code") or "")[:3].upper()
            matched_flag = ICD_TO_CHRONIC_FLAG.get(code_prefix)
            if matched_flag in flags:
                flags[matched_flag] = 1

    return flags


def _get_provider_stats(provider_id: str) -> dict:
    """
    Reads the precomputed Day 3 provider lookup table.
    Falls back to global defaults if provider is missing.
    """
    try:
        row = (
            spark.table(PROVIDER_LOOKUP_TABLE)
            .filter(F.col("Provider") == provider_id)
            .select(
                "ProviderClaimCount",
                "ProviderAvgClaimAmt",
                "ProviderClaimAmtStdDev",
                "ProviderAvgDiagnosisCount",
                "ProviderAvgLOS",
            )
            .first()
        )
        if row:
            return row.asDict()
    except Exception as e:
        print(f"[WARN] Provider lookup read failed: {e}")

    print(
        f"[WARN] Provider '{provider_id}' not found in lookup table. Using fallback averages.")
    return {
        "ProviderClaimCount": 50,
        "ProviderAvgClaimAmt": 5000.0,
        "ProviderClaimAmtStdDev": 2000.0,
        "ProviderAvgDiagnosisCount": 4.0,
        "ProviderAvgLOS": 5.0,
    }


def get_patient_from_fhir(patient_id: str) -> dict:
    """
    Tool 1: Fetch Patient + Condition + Claim and map to the model feature vector.
    """
    print(f"[Tool 1] Fetching FHIR resources for patient_id={patient_id}")

    # ── Patient ────────────────────────────────────────────────────────────────
    patient = _fhir_get("Patient", resource_id=patient_id)
    birth_date = patient.get("birthDate", "1950-01-01")
    gender_raw = (patient.get("gender") or "unknown").lower()
    gender = 1 if gender_raw == "male" else 0

    # ── Conditions ─────────────────────────────────────────────────────────────
    condition_bundle = _fhir_get(
        "Condition", params={"patient": patient_id, "_count": 200})
    conditions = [e["resource"] for e in condition_bundle.get("entry", [])]
    chronic_flags = _map_chronic_conditions(conditions)
    chronic_condition_count = int(sum(chronic_flags.values()))
    renal_disease = int(chronic_flags.get("ChronicCond_KidneyDisease", 0))

    # ── Most recent claim ──────────────────────────────────────────────────────
    claim_bundle = _fhir_get(
        "Claim",
        params={"patient": patient_id, "_sort": "-created", "_count": 1},
    )
    entries = claim_bundle.get("entry", [])
    if not entries:
        raise ValueError(
            f"No Claim resources found for patient_id={patient_id}")

    claim = entries[0]["resource"]

    # ── Claim fields ───────────────────────────────────────────────────────────
    claim_amount = float(claim.get("total", {}).get("value", 0.0) or 0.0)
    deductible = 0.0  # standard FHIR Claim usually doesn't give your Kaggle deductible field

    claim_created = (claim.get("created") or date.today().isoformat())[:10]
    claim_start = datetime.strptime(claim_created, "%Y-%m-%d").date()
    claim_end = claim_start  # no true end date in your current mapping
    claim_duration = max(1, (claim_end - claim_start).days)

    # Diagnose codes
    diagnoses = [
        d.get("diagnosisCodeableConcept", {}).get(
            "coding", [{}])[0].get("code", "UNKNOWN")
        for d in claim.get("diagnosis", [])
    ]
    diagnosis_codes = {
        f"ClmDiagnosisCode_{i+1}": diagnoses[i] if i < len(diagnoses) else None
        for i in range(9)
    }
    diagnosis_code_count = len([d for d in diagnoses if d and d != "UNKNOWN"])
    primary_dx = diagnosis_codes.get("ClmDiagnosisCode_1") or "UNKNOWN"
    has_primary_diagnosis = int(primary_dx != "UNKNOWN")
    icd_chapter = primary_dx[:3] if len(primary_dx) >= 3 else "UNK"

    # Procedure codes
    procedures = [
        p.get("procedureCodeableConcept", {}).get(
            "coding", [{}])[0].get("code")
        for p in claim.get("procedure", [])
    ]
    procedure_code_count = len([p for p in procedures if p])
    has_operating_physician = int(procedure_code_count > 0)

    # Claim type heuristic
    claim_type_coding = claim.get("type", {}).get(
        "coding", [{}])[0].get("code", "")
    claim_type = 1 if "institutional" in claim_type_coding.lower() else 0

    # Provider
    provider_ref = claim.get("provider", {}).get(
        "reference", "UNKNOWN_PROVIDER")
    provider_id = provider_ref.split("/")[-1]

    # Provider aggregates
    provider_stats = _get_provider_stats(provider_id)
    std_dev = float(provider_stats.get("ProviderClaimAmtStdDev", 0.0) or 0.0)
    provider_avg_claim_amt = float(
        provider_stats.get("ProviderAvgClaimAmt", 0.0) or 0.0)
    z_score = ((claim_amount - provider_avg_claim_amt) /
               std_dev) if std_dev > 0 else 0.0

    # IMPORTANT: age relative to claim date, not today's date
    age = _compute_age(birth_date, claim_start)

    feature_vector = {
        # Tracking-only identifiers
        "_patient_id": patient_id,
        "_provider_id": provider_id,

        # Numeric
        "InscClaimAmtReimbursed": float(claim_amount),
        "DeductibleAmtPaid": float(deductible),
        "ClaimDuration": int(claim_duration),
        "LengthOfStay": None,  # kept as None; imputed later
        "Age": int(age),
        "DiagnosisCodeCount": int(diagnosis_code_count),
        "ProcedureCodeCount": int(procedure_code_count),
        "PhysicianCount": int(has_operating_physician + 1),
        "ChronicConditionCount": int(chronic_condition_count),
        "MedicareCoverageMonths": 24,
        "IPtoOPReimbursementRatio": 1.0,
        "ProviderClaimCount": int(provider_stats["ProviderClaimCount"]),
        "ProviderAvgClaimAmt": float(provider_stats["ProviderAvgClaimAmt"]),
        "ClaimAmtZScore": float(round(z_score, 4)),

        # Binary
        "claim_type": int(claim_type),
        "IsDeceased": 0,
        "HasRenalDisease": int(renal_disease),
        "HasOperatingPhysician": int(has_operating_physician),
        "HasOtherPhysician": 0,
        "HasPrimaryDiagnosis": int(has_primary_diagnosis),

        # Categorical
        "Gender": int(gender),
        "Race": 3,
        "State": "UNK",
        "ICD_Chapter": icd_chapter,

        # Prompt context
        "_chronic_flags": chronic_flags,
        "_primary_diagnosis": primary_dx,
        "_claim_amount": float(claim_amount),
        "_claim_duration": int(claim_duration),
        "_claim_start_date": claim_start.isoformat(),
    }

    print(
        f"[Tool 1] Feature vector assembled. ClaimAmtZScore={z_score:.2f}, Age={age}")
    return feature_vector

4 · Tool 2 — `predict_fraud_risk`

Loads the registered ML model from MLflow Model Registry, builds the feature
DataFrame in the exact column order used during training, runs prediction,
and computes SHAP values.

**`MOCK_MODE = True`** until Day 3 registers the model. Flip the flag in Cell 1.


In [ ]:
import os
import json
import tempfile
import pandas as pd
import numpy as np
import mlflow
import shap


UC_MODEL_NAME = f"{UC_CATALOG}.{UC_SCHEMA}.{ML_MODEL_REGISTRY_NAME}"
UC_MODEL_URI = f"models:/{UC_MODEL_NAME}@Champion"


def _get_champion_model_version():
    client = mlflow.tracking.MlflowClient()
    mv = client.get_model_version_by_alias(UC_MODEL_NAME, "Champion")
    if mv is None:
        raise RuntimeError(
            f"No model alias 'Champion' found for {UC_MODEL_NAME}")
    return mv


def _load_feature_columns() -> list[str]:
    """
    Load feature_columns.json from the SAME run as the Champion model version.
    Falls back to the agreed base columns if artifact is unavailable.
    """
    try:
        mv = _get_champion_model_version()
        client = mlflow.tracking.MlflowClient()

        with tempfile.TemporaryDirectory() as tmpdir:
            local_path = client.download_artifacts(
                run_id=mv.run_id,
                path="feature_columns.json",
                dst_path=tmpdir,
            )
            with open(local_path, "r") as f:
                cols = json.load(f)

        if not isinstance(cols, list) or not cols:
            raise ValueError("feature_columns.json is empty or invalid")

        return cols

    except Exception as e:
        print(
            f"[WARN] Could not load feature_columns.json from Champion run: {e}")

    # Fallback base order. OHE columns will be added dynamically if present in trained artifact.
    return [
        "InscClaimAmtReimbursed",
        "DeductibleAmtPaid",
        "ClaimDuration",
        "LengthOfStay",
        "Age",
        "DiagnosisCodeCount",
        "ProcedureCodeCount",
        "PhysicianCount",
        "ChronicConditionCount",
        "MedicareCoverageMonths",
        "IPtoOPReimbursementRatio",
        "ProviderClaimCount",
        "ProviderAvgClaimAmt",
        "ClaimAmtZScore",
        "claim_type",
        "IsDeceased",
        "HasRenalDisease",
        "HasOperatingPhysician",
        "HasOtherPhysician",
        "HasPrimaryDiagnosis",
        "Gender",
        "Race",
    ]


def _prepare_inference_df(feature_vector: dict, feature_cols: list[str]) -> pd.DataFrame:
    """
    Minimal inference preprocessing.
    Assumes Day 3 training used the same base schema and OHE contract.
    """
    row = {k: v for k, v in feature_vector.items() if not k.startswith("_")}

    if row.get("LengthOfStay") is None:
        row["LengthOfStay"] = 0

    df = pd.DataFrame([row])

    # One-hot encode only the fields you explicitly use that way
    for col in ["State", "ICD_Chapter"]:
        if col not in df.columns:
            df[col] = "UNK"

    df = pd.get_dummies(df, columns=["State", "ICD_Chapter"], dtype=int)

    # Add any missing trained columns as zero
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0

    # Drop unexpected extra columns and enforce exact order
    df = df[feature_cols]

    return df


def _extract_positive_class_shap(model, X: pd.DataFrame) -> dict[str, float]:
    """
    Returns SHAP values for the positive class in a consistent dict format.
    Works for common tree and linear sklearn classifiers.
    """
    # Tree-based models
    if hasattr(model, "estimators_") or hasattr(model, "tree_"):
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X)

        # Binary tree models may return:
        # - list[class0, class1]
        # - array shape (n, features)
        # - array shape (n, features, classes)
        if isinstance(shap_values, list):
            sv = shap_values[1][0] if len(
                shap_values) > 1 else shap_values[0][0]
        else:
            arr = np.array(shap_values)
            if arr.ndim == 3:
                sv = arr[0, :, 1] if arr.shape[-1] > 1 else arr[0, :, 0]
            elif arr.ndim == 2:
                sv = arr[0]
            else:
                raise ValueError(
                    f"Unexpected SHAP output shape for tree model: {arr.shape}")

        return {col: float(val) for col, val in zip(X.columns, sv)}

    # Linear / generalized linear models
    if hasattr(model, "coef_"):
        background = np.zeros((1, X.shape[1]), dtype=float)
        explainer = shap.LinearExplainer(model, background)
        shap_values = explainer.shap_values(X)

        if isinstance(shap_values, list):
            sv = shap_values[1][0] if len(
                shap_values) > 1 else shap_values[0][0]
        else:
            arr = np.array(shap_values)
            if arr.ndim == 2:
                sv = arr[0]
            elif arr.ndim == 3:
                sv = arr[0, :, 1] if arr.shape[-1] > 1 else arr[0, :, 0]
            else:
                raise ValueError(
                    f"Unexpected SHAP output shape for linear model: {arr.shape}")

        return {col: float(val) for col, val in zip(X.columns, sv)}

    raise TypeError(
        f"Unsupported model type for SHAP explanation: {type(model)}")


def predict_fraud_risk(feature_vector: dict) -> dict:
    """
    Tool 2: Run the trained model against the feature vector.
    """
    if MOCK_MODE:
        print("[Tool 2] MOCK_MODE=True — returning synthetic prediction.")
        return {
            "risk_score": 0.84,
            "shap_values": {
                "ClaimAmtZScore": 0.31,
                "InscClaimAmtReimbursed": 0.22,
                "DiagnosisCodeCount": -0.14,
                "ChronicConditionCount": 0.11,
                "LengthOfStay": 0.09,
            },
            "ml_run_id": "mock_run_000",
        }

    # 1) Load champion model from UC alias
    model = mlflow.sklearn.load_model(UC_MODEL_URI)

    # 2) Get linked training run
    mv = _get_champion_model_version()
    ml_run_id = mv.run_id

    # 3) Prepare inference frame
    feature_cols = _load_feature_columns()
    X = _prepare_inference_df(feature_vector, feature_cols)

    # 4) Predict
    if not hasattr(model, "predict_proba"):
        raise AttributeError("Loaded model does not expose predict_proba().")

    risk_score = float(model.predict_proba(X)[0][1])

    # 5) SHAP
    shap_dict = _extract_positive_class_shap(model, X)

    return {
        "risk_score": risk_score,
        "shap_values": shap_dict,
        "ml_run_id": ml_run_id,
    }

5 · LLM Prompt Construction

Prompt template is versioned (logged to MLflow as a parameter).
The SHAP formatter translates raw feature names into plain English
before the LLM ever sees them — this is the primary hallucination guardrail.


In [ ]:
#  ── Human-readable labels for SHAP features ──────────────────────────────────
FEATURE_LABELS = {
    "ClaimAmtZScore":             "Claim amount deviation from provider average (Z-score)",
    "InscClaimAmtReimbursed":     "Total claim reimbursement amount ($)",
    "DiagnosisCodeCount":         "Number of diagnosis codes on claim",
    "ChronicConditionCount":      "Number of chronic conditions (patient)",
    "LengthOfStay":               "Inpatient length of stay (days)",
    "ProviderClaimCount":         "Total claims submitted by this provider",
    "ProviderAvgClaimAmt":        "Provider's average claim amount ($)",
    "ClaimDuration":              "Claim service duration (days)",
    "Age":                        "Patient age",
    "HasPrimaryDiagnosis":        "Primary diagnosis code present (1=Yes, 0=No)",
    "HasOperatingPhysician":      "Operating physician listed (1=Yes, 0=No)",
    "DeductibleAmtPaid":          "Deductible amount paid ($)",
    "MedicareCoverageMonths":     "Total Medicare coverage months (Parts A+B)",
    "IPtoOPReimbursementRatio":   "Inpatient-to-outpatient reimbursement ratio",
    "claim_type":                 "Claim type (1=Inpatient, 0=Outpatient)",
    "HasRenalDisease":            "Renal disease flag (1=Yes)",
    "IsDeceased":                 "Patient deceased at claim time (1=Yes)",
}


def format_shap_for_prompt(shap_values: dict, top_n: int = TOP_N_SHAP) -> str:
    """
    Picks the top N SHAP features by absolute value and formats them
    as a readable, signed bullet list for the LLM prompt.

    Example output:
        - Claim amount deviation from provider average (Z-score): +0.31
          (this claim's amount is significantly above the provider's normal billing)
        - Total claim reimbursement amount ($): +0.22
          (unusually high reimbursement relative to similar claims)
        - Number of diagnosis codes on claim: -0.14
          (fewer diagnosis codes than expected — may indicate undercoding)
    """
    top = sorted(shap_values.items(), key=lambda x: abs(
        x[1]), reverse=True)[:top_n]
    lines = []
    for feature, value in top:
        label = FEATURE_LABELS.get(feature, feature)
        direction = "increased" if value > 0 else "decreased"
        magnitude = "significantly" if abs(value) > 0.2 else "moderately"
        sign = "+" if value > 0 else ""
        lines.append(
            f"  - {label}: {sign}{value:.3f}\n"
            f"    (This factor {magnitude} {direction} the anomaly risk score)"
        )
    return "\n".join(lines)


 #  ── Prompt templates ──────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a healthcare claims auditor AI assistant.
Your role is to explain why a medical claim has been flagged as high anomaly/audit risk,
based ONLY on the structured data provided to you.

STRICT RULES — violations make the output unusable:
1. Do NOT mention any diagnosis, condition, medication, or clinical detail that is not explicitly present in the input data.
2. Do NOT speculate about patient health beyond the chronic condition flags provided.
3. Frame all findings as "anomalous billing patterns that historically trigger payer audits and systemic claim denials" — NOT as proof of fraud.
4. Output ONLY a single valid JSON object. No preamble, no explanation, no markdown fences.
5. All three keys (denial_reasons, corrective_actions, risk_summary) are required.
"""

USER_PROMPT_TEMPLATE = """Analyze the following claim and generate a structured JSON explanation.

=== PATIENT CONTEXT ===
Age: {age} years
Active Chronic Conditions: {chronic_conditions}
Medicare Coverage: {medicare_coverage_months} months (Parts A+B combined)

=== CLAIM CONTEXT ===
Claim Type: {claim_type}
Claim Amount: ${claim_amount:,.2f}
Claim Duration: {claim_duration} days
Diagnosis Code Count: {diagnosis_code_count}
Primary Diagnosis Chapter (ICD-10): {icd_chapter}

=== MODEL OUTPUT ===
Anomaly Risk Score: {risk_score:.2f} (scale 0.0–1.0; threshold for high risk: 0.5)
Risk Interpretation: {"HIGH RISK — recommend pre-submission review" if {risk_score} >= 0.5 else "LOW RISK"}

Top {top_n} Contributing Factors (SHAP feature importance values):
{shap_summary}

=== REQUIRED OUTPUT FORMAT ===
Return exactly this JSON schema — no extra keys, no missing keys:
{{
  "denial_reasons": [
    "Specific reason 1 grounded in the data above",
    "Specific reason 2 grounded in the data above",
    "Specific reason 3 grounded in the data above"
  ],
  "corrective_actions": [
    "Actionable step 1 the billing team can take before submission",
    "Actionable step 2",
    "Actionable step 3"
  ],
  "risk_summary": "A 2–3 sentence plain-English summary for a billing reviewer explaining the overall anomaly risk and what to do next."
}}"""


def build_prompt(feature_vector: dict, prediction: dict) -> tuple[str, str]:
    """
    Assembles the system + user prompt strings from a feature vector and
    prediction result. Returns (system_prompt, user_prompt).
    """
    chronic_conditions = [
        flag.replace("ChronicCond_", "").replace("_", " ")
        for flag, val in feature_vector.get("_chronic_flags", {}).items()
        if val == 1
    ] or ["None documented"]

    claim_type_label = "Inpatient" if feature_vector.get(
        "claim_type") == 1 else "Outpatient"
    shap_summary = format_shap_for_prompt(prediction["shap_values"])
    risk_score = prediction["risk_score"]

    user_prompt = USER_PROMPT_TEMPLATE.format(
        age=feature_vector.get("Age", "Unknown"),
        chronic_conditions=", ".join(chronic_conditions),
        medicare_coverage_months=feature_vector.get(
            "MedicareCoverageMonths", "N/A"),
        claim_type=claim_type_label,
        claim_amount=feature_vector.get("_claim_amount", 0.0),
        claim_duration=feature_vector.get("_claim_duration", 0),
        diagnosis_code_count=feature_vector.get("DiagnosisCodeCount", 0),
        icd_chapter=feature_vector.get("ICD_Chapter", "Unknown"),
        risk_score=risk_score,
        top_n=TOP_N_SHAP,
        shap_summary=shap_summary,
    )

    return SYSTEM_PROMPT, user_prompt

6 · LLM Call

Uses `databricks-openai` for auto-configured workspace auth.
`mlflow.openai.autolog()` (set in Cell 2) captures token counts and
latency automatically — no manual metric extraction needed.


In [ ]:
import json
import time
from databricks.sdk import WorkspaceClient
from openai import OpenAI as DatabricksOpenAI


def _get_databricks_openai_client() -> DatabricksOpenAI:
    """
    OpenAI-compatible client pointed at the Databricks workspace.
    """
    w = WorkspaceClient()
    return DatabricksOpenAI(
        base_url=f"{DATABRICKS_HOST}/serving-endpoints",
        api_key=w.config.token,
    )


def call_llm(feature_vector: dict, prediction: dict) -> dict:
    """
    Calls the Databricks-hosted LLM and returns parsed structured JSON.
    """
    system_prompt, user_prompt = build_prompt(feature_vector, prediction)
    client = _get_databricks_openai_client()

    response_format = {
        "type": "json_schema",
        "json_schema": {
            "name": "denial_explanation",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "denial_reasons": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                    "corrective_actions": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                    "risk_summary": {
                        "type": "string",
                    },
                },
                "required": [
                    "denial_reasons",
                    "corrective_actions",
                    "risk_summary",
                ],
                "additionalProperties": False,
            },
        },
    }

    t0 = time.time()
    response = client.chat.completions.create(
        model=LLM_ENDPOINT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=800,
        response_format=response_format,
    )
    latency_ms = int((time.time() - t0) * 1000)

    raw_content = response.choices[0].message.content

    # Some clients return a JSON string; keep parser strict
    if not isinstance(raw_content, str):
        raise ValueError(f"Unexpected LLM content type: {type(raw_content)}")

    try:
        parsed = json.loads(raw_content)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"LLM returned invalid JSON.\n"
            f"Raw response: {raw_content[:500]}\n"
            f"JSON error: {e}"
        )

    required_keys = {"denial_reasons", "corrective_actions", "risk_summary"}
    missing = required_keys - set(parsed.keys())
    if missing:
        raise ValueError(f"LLM response missing required keys: {missing}")

    parsed["_latency_ms"] = latency_ms
    parsed["_input_tokens"] = getattr(response.usage, "prompt_tokens", 0)
    parsed["_output_tokens"] = getattr(response.usage, "completion_tokens", 0)
    parsed["_model_used"] = LLM_ENDPOINT

    print(
        f"[LLM] Response received. latency={latency_ms}ms, "
        f"tokens={getattr(response.usage, 'total_tokens', 'n/a')}, "
        f"risk_score={prediction['risk_score']:.2f}"
    )

    return parsed

7 · MLflow LLM Experiment Logging

Logs every LLM inference run as a separate MLflow experiment entry,
linked back to the ML model run via the `ml_run_id` tag.
Also writes the prediction to the Delta predictions table for the dashboard.


In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, TimestampType
from datetime import datetime as dt


def log_and_sink(
    patient_id: str,
    feature_vector: dict,
    prediction: dict,
    llm_response: dict,
) -> str:
    """
    1. Logs the LLM run as a named MLflow experiment entry.
    2. Writes a prediction record to the Delta predictions table.
    Returns the MLflow run_id for reference.
    """
    patient_id = feature_vector.get("_patient_id", patient_id)
    provider_id = feature_vector.get("_provider_id", "UNKNOWN")
    risk_score = prediction["risk_score"]
    ml_run_id = prediction.get("ml_run_id", "unknown")
    top_shap = dict(sorted(
        prediction["shap_values"].items(),
        key=lambda x: abs(x[1]), reverse=True
    )[:TOP_N_SHAP])

    with mlflow.start_run(run_name=f"llm_explanation_{patient_id[:8]}") as run:

        #  ── Parameters ────────────────────────────────────────────────────────
        mlflow.log_params({
            "prompt_template_version": PROMPT_TEMPLATE_VERSION,
            "llm_endpoint":            LLM_ENDPOINT,
            "top_n_shap":              TOP_N_SHAP,
            "mock_mode":               str(MOCK_MODE),
        })

        #  ── Metrics ───────────────────────────────────────────────────────────
        mlflow.log_metrics({
            "risk_score":     risk_score,
            "latency_ms":     llm_response["_latency_ms"],
            "input_tokens":   llm_response["_input_tokens"],
            "output_tokens":  llm_response["_output_tokens"],
            "total_tokens":   llm_response["_input_tokens"] + llm_response["_output_tokens"],
        })

        #  ── Artifacts ─────────────────────────────────────────────────────────
        mlflow.log_dict(top_shap, "shap_values_used.json")
        mlflow.log_dict({
            "denial_reasons":    llm_response["denial_reasons"],
            "corrective_actions": llm_response["corrective_actions"],
            "risk_summary":      llm_response["risk_summary"],
        }, "llm_response_output.json")

        #  ── Tags (link back to ML run) ────────────────────────────────────────
        mlflow.set_tags({
            "patient_id":  patient_id,
            "provider_id": provider_id,
            "ml_run_id":   ml_run_id,
        })

        run_id = run.info.run_id

    #  ── Delta sink (for analytics dashboard) ─────────────────────────────────
    record = {
        "run_id":             run_id,
        "patient_id":         patient_id,
        "provider_id":        provider_id,
        "risk_score":         float(risk_score),
        "claim_amount":       float(feature_vector.get("_claim_amount", 0.0)),
        "claim_duration":     int(feature_vector.get("_claim_duration", 0)),
        "top_shap_feature_1": list(top_shap.keys())[0] if top_shap else None,
        "top_shap_value_1":   float(list(top_shap.values())[0]) if top_shap else None,
        "denial_reasons":     json.dumps(llm_response["denial_reasons"]),
        "corrective_actions": json.dumps(llm_response["corrective_actions"]),
        "risk_summary":       llm_response["risk_summary"],
        "prediction_ts":      dt.utcnow().isoformat(),
        "prompt_version":     PROMPT_TEMPLATE_VERSION,
        "mock_mode":          MOCK_MODE,
    }

    sdf = spark.createDataFrame([record])
    (sdf.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(PREDICTIONS_TABLE))

    print(f"[MLflow] Run logged: {run_id}")
    print(f"[Delta]  Record appended to {PREDICTIONS_TABLE}")
    return run_id

8 · ChatAgent — AI Playground Deployment

Wraps the full pipeline (Tool1 → Tool2 → LLM → MLflow log) as an
`mlflow.pyfunc.ChatAgent` for deployment to Databricks Model Serving
and interactive use in the AI Playground.


In [ ]:
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse, ChatAgentChunk


class DenialPreventionAgent(ChatAgent):
    """
    Conversational agent that accepts a FHIR patient ID via chat and returns
    an anomaly risk explanation with corrective actions.

    Expected user message format (any of these work):
        "Analyze patient a7b3c9d2-1234-5678-abcd-ef0123456789"
        "a7b3c9d2-1234-5678-abcd-ef0123456789"
        "Patient ID: a7b3c9d2-..."
    """

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context=None,
        custom_inputs=None,
    ) -> ChatAgentResponse:

        #  ── Extract patient ID from the last user message ─────────────────────
        last_user_msg = next(
            (m["content"] for m in reversed(messages) if m["role"] == "user"),
            ""
        )
        patient_id = self._extract_patient_id(last_user_msg)

        if not patient_id:
            return ChatAgentResponse(messages=[{
                "role": "assistant",
                "content": (
                    "Please provide a valid FHIR patient ID to analyze.\n\n"
                    "Example: `Analyze patient a7b3c9d2-1234-5678-abcd-ef0123456789`"
                )
            }])

        #  ── Run the full pipeline ─────────────────────────────────────────────
        try:
            feature_vector = get_patient_from_fhir(patient_id)
            prediction = predict_fraud_risk(feature_vector)
            llm_response = call_llm(feature_vector, prediction)
            run_id = log_and_sink(
                patient_id, feature_vector, prediction, llm_response)

            reply = self._format_chat_response(
                patient_id, prediction, llm_response, run_id)

        except Exception as e:
            reply = (
                f"⚠️ Error processing patient `{patient_id}`:\n```\n{str(e)}\n```\n\n"
                "Please verify the patient ID exists on the FHIR server and try again."
            )

        return ChatAgentResponse(messages=[{"role": "assistant", "content": reply}])

    #  ── Helpers ───────────────────────────────────────────────────────────────

    def _extract_patient_id(self, text: str) -> str:
        """
        Extracts a UUID-style patient ID from free text.
        Handles: bare UUIDs, "patient <id>", "Patient ID: <id>", etc.
        """
        import re
        #  Standard UUID pattern
        uuid_pattern = r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}"
        match = re.search(uuid_pattern, text)
        if match:
            return match.group(0)
        #  Fallback: treat the whole stripped input as an ID (for non-UUID HAPI IDs)
        cleaned = text.strip().split()[-1]
        return cleaned if len(cleaned) > 4 else ""

    def _format_chat_response(
        self,
        patient_id: str,
        prediction: dict,
        llm_response: dict,
        run_id: str,
    ) -> str:
        """Formats the agent reply for the AI Playground chat interface."""
        risk = prediction["risk_score"]
        risk_icon = "🔴" if risk >= 0.7 else "🟡" if risk >= 0.5 else "🟢"
        level = "HIGH" if risk >= 0.7 else "MEDIUM" if risk >= 0.5 else "LOW"

        reasons = "\n".join(f"  {i+1}. {r}" for i,
                            r in enumerate(llm_response["denial_reasons"]))
        actions = "\n".join(
            f"  {i+1}. {a}" for i, a in enumerate(llm_response["corrective_actions"]))

        return f""" Claim Anomaly Risk Report
**Patient ID:** `{patient_id}`
**Anomaly Risk Score:** {risk:.2f} / 1.00  {risk_icon} **{level} RISK**

---

 Summary
{llm_response["risk_summary"]}

---

 Why This Claim Was Flagged
{reasons}

---

 Recommended Actions Before Submission
{actions}

---
*MLflow run ID: `{run_id}` · Model: `{LLM_ENDPOINT}` · Prompt: `{PROMPT_TEMPLATE_VERSION}`*
*⚠️ Note: Risk score is based on anomalous billing pattern detection, not a direct denial prediction.*"""

9 · Register Agent to MLflow Model Registry

Run this cell once to log and register the agent.
After registration, go to **Databricks UI → Serving → Create Endpoint**
and select the registered model to enable it in AI Playground.


In [ ]:
import os
import mlflow

# Recommended for some UC artifact-repo permission edge cases
os.environ["MLFLOW_USE_DATABRICKS_SDK_MODEL_ARTIFACTS_REPO_FOR_UC"] = "True"

AGENT_UC_NAME = f"{UC_CATALOG}.{UC_SCHEMA}.denial_prevention_agent"

with mlflow.start_run(run_name="denial_prevention_agent_registration"):
    agent_info = mlflow.pyfunc.log_model(
        name="denial_prevention_agent",
        python_model=DenialPreventionAgent(),
        pip_requirements=[
            "openai",
            "databricks-sdk",
            "shap",
            "requests",
            "mlflow",
            "pandas",
            "numpy",
        ],
        input_example={
            "messages": [
                {
                    "role": "user",
                    "content": "Analyze patient a7b3c9d2-1234-5678-abcd-ef0123456789",
                }
            ]
        },
    )

registered = mlflow.register_model(
    model_uri=agent_info.model_uri,
    name=AGENT_UC_NAME,
)

print(f"Agent registered: {AGENT_UC_NAME} v{registered.version}")
print(f"Model URI: {agent_info.model_uri}")
print()
print("Next steps:")
print("  1. Databricks UI → Serving → Create Endpoint")
print(f"  2. Entity: {AGENT_UC_NAME}  Version: {registered.version}")
print("  3. Workload size: Small")
print("  4. Once endpoint is running → AI Playground → select endpoint → chat")

10 · End-to-End Test (Test Cases 3 & 4)

Run this cell to validate the full pipeline end-to-end.
Replace `TEST_PATIENT_ID` with a real patient ID you POSTed to HAPI FHIR.


In [ ]:
#  Replace with a real patient ID from your Synthea FHIR upload
TEST_PATIENT_ID = "REPLACE_WITH_REAL_SYNTHEA_PATIENT_ID"

print("=" * 60)
print("END-TO-END PIPELINE TEST")
print("=" * 60)

#  Step 1 — Tool 1
print("\n[Step 1] Tool 1: get_patient_from_fhir")
feature_vector = get_patient_from_fhir(TEST_PATIENT_ID)
print(f"  Age={feature_vector['Age']}, "
      f"ClaimAmt=${feature_vector['InscClaimAmtReimbursed']:,.2f}, "
      f"ZScore={feature_vector['ClaimAmtZScore']:.2f}, "
      f"ChronicCount={feature_vector['ChronicConditionCount']}")

#  Step 2 — Tool 2
print("\n[Step 2] Tool 2: predict_fraud_risk")
prediction = predict_fraud_risk(feature_vector)
print(f"  risk_score={prediction['risk_score']:.3f}")
print(
    f"  top SHAP: {dict(list(sorted(prediction['shap_values'].items(), key=lambda x: abs(x[1]), reverse=True))[:3])}")

#  Step 3 — LLM
print("\n[Step 3] LLM call")
llm_response = call_llm(feature_vector, prediction)
print(
    f"  latency={llm_response['_latency_ms']}ms, tokens={llm_response['_input_tokens']+llm_response['_output_tokens']}")
print(f"  risk_summary: {llm_response['risk_summary'][:120]}...")

#  Step 4 — MLflow log + Delta sink
print("\n[Step 4] MLflow logging + Delta sink")
run_id = log_and_sink(TEST_PATIENT_ID, feature_vector,
                      prediction, llm_response)

#  Test Case 3 validation: schema check
assert "denial_reasons" in llm_response, "FAIL: missing denial_reasons"
assert "corrective_actions" in llm_response, "FAIL: missing corrective_actions"
assert "risk_summary" in llm_response, "FAIL: missing risk_summary"
assert isinstance(llm_response["denial_reasons"],
                  list), "FAIL: denial_reasons not a list"
assert len(llm_response["denial_reasons"]) > 0, "FAIL: denial_reasons is empty"

print("\n✅ Test Case 3 PASSED — LLM returned valid JSON with required schema")
print("✅ Test Case 4 PASSED — full pipeline ran end-to-end from patient_id")
print(f"\nMLflow run: {run_id}")
print(f"Delta table: {PREDICTIONS_TABLE}")
print("=" * 60)

11 · Setup Helper — POST Synthea Patients to HAPI FHIR

Run this cell once to upload your Synthea JSON bundles to the public HAPI FHIR sandbox.
Save the returned patient IDs for use in the AI Playground demo.


In [ ]:
import os
import glob
import json
import requests
import time


def upload_synthea_bundle(bundle_path: str, max_retries: int = 3) -> list[tuple[str, str]]:
    """
    POST a Synthea FHIR Bundle JSON file to the HAPI FHIR server.
    Returns [(resource_type, resource_id), ...].
    """
    with open(bundle_path, "r") as f:
        bundle = json.load(f)

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.post(
                FHIR_BASE_URL,
                json=bundle,
                headers={
                    "Content-Type": "application/fhir+json",
                    "Accept": "application/fhir+json",
                },
                timeout=60,
            )

            if resp.status_code in {429, 500, 502, 503, 504}:
                raise requests.HTTPError(
                    f"Transient upload error {resp.status_code}: {resp.text[:300]}",
                    response=resp,
                )

            resp.raise_for_status()
            result = resp.json()

            created = []
            for entry in result.get("entry", []):
                location = entry.get("response", {}).get("location", "")
                if location:
                    parts = location.rstrip("/").split("/")
                    if len(parts) >= 2:
                        created.append((parts[-2], parts[-1]))

            return created

        except Exception as e:
            last_err = e
            if attempt == max_retries:
                break
            sleep_s = 2 ** attempt
            print(
                f"[WARN] Upload failed on attempt {attempt}/{max_retries}: {e}")
            time.sleep(sleep_s)

    raise RuntimeError(
        f"Bundle upload failed after {max_retries} attempts: {last_err}")


# ── Example usage ──────────────────────────────────────────────────────────────
SYNTHEA_OUTPUT_DIR = "/dbfs/FileStore/synthea/output/fhir"

patient_ids = []

for bundle_file in glob.glob(f"{SYNTHEA_OUTPUT_DIR}/*.json")[:10]:
    print(f"Uploading {os.path.basename(bundle_file)}...")
    try:
        created = upload_synthea_bundle(bundle_file)
        for rtype, rid in created:
            if rtype == "Patient":
                patient_ids.append(rid)
                print(f"  ✅ Patient created: {rid}")
    except Exception as e:
        print(f"  ❌ Failed: {e}")

print(f"\nTotal patients uploaded: {len(patient_ids)}")
print("Patient IDs for demo:")
for pid in patient_ids:
    print(f"  {pid}")